# Feed the Beast
### PyLadies Amsterdam × Xomnia · 25 June 2025

---

Seven exercises on feeding your LLM the best data.

| Exercise | Problem | What you build |
|----------|---------|----------------|
| **1 — Scrub before you send** | Support tickets contain PII you cannot send to an external API | Regex pass → LLM detection → combined scrubber |
| **2 — Minimum viable context** | Sending everything is expensive and slow. How little context can you get away with? | Progressive compression + token tracking |
| **3 — Right context, right question** | A wide table has 15 columns; most are irrelevant to any given question | LLM column selection vs your pick vs send-all |
| **4 — When the LLM transforms the data** | Messy free-text listings need structured columns downstream | Extraction prompt → schema validation → retry loop |
| **5 — Numbers they can trust** | A manager asks for revenue totals; the LLM invents or miscalculates when it does the math | Pre-compute verified facts → LLM explains only |

To spark your brain after the workshop, there are some additional exercises that can be done
| Exercise | Problem | What you build |
|----------|---------|----------------|
| **6 — Signal that survives** | Map-reduce summarisation hides class imbalance — every batch counts equally | Frequency count → proportional sample → counts in the prompt |
| **7 — Speak their language** | Business users ask about "churn" and "active subscribers"; your warehouse uses cryptic column names | Business glossary in the prompt → correct column mapping |

**Cell legend:**  
▶ `RUN` — run the cell and observe the output  
🎯 `YOUR TURN` — write or fix code yourself  
💬 `DISCUSS` — talk with the person next to you

---
## Setup ▶ `RUN`

In [ ]:
import json
import os
import re
import sqlite3

import numpy as np
import pandas as pd
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",

    # fill in: api_key = "sk-or-v1-..."
    api_key=os.environ["OPENROUTER_API_KEY"],
)
MODEL = "gpt-4o-mini"

def call_llm(prompt: str, max_tokens: int = 600) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()

def call_llm_tracked(prompt: str, max_tokens: int = 600) -> tuple[str, int]:
    """Returns (answer, input_token_count)."""
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip(), response.usage.prompt_tokens

print("Setup complete ✓")

print(call_llm("Say hello world!", 50))
print("Test complete ✓")

---
## Exercise 1 — Scrub before you send: removing PII from data ⏱️ 10 min

Your analytics pipeline processes customer support tickets and sends them to an external LLM API
to extract themes and sentiment. The tickets contain names, email addresses, phone numbers.

Sending this as-is violates your data processing agreement and almost certainly GDPR.

**The challenge:** remove personal data before the API call without destroying
the context the LLM needs to answer correctly.

Two failure modes:
- **Under-scrub:** names embedded in sentences slip through regex → data leak
- **Over-scrub:** replacing too aggressively leaves the LLM with incoherent fragments

You will build a two-pass scrubber: regex for format-defined PII (fast, free),
then LLM detection for context-defined PII (names in sentences) on what remains.



In [ ]:
# ▶ RUN — support tickets containing PII

tickets_pii = [
    "Hi, I am Maria Jansen (m.jansen@gmail.com). Order 45821 is 5 days late.",
    "Jan de Vries here, tel 06-12345678. My refund for order 45891 is missing.",
    "The product I bought for my daughter Emma broke after 2 days of use.",
    "Please reach me at info@debruijn-company.nl or call +31 6 12 34 56 78.",
    "My name is listed wrong on invoice 56781. It says Peter instead of Pieter.",
    "I, Sophie van der Berg, demand a full refund. Order 78234 arrived damaged.",
    "Account email j.bakker@hotmail.com no longer works. Cannot log in.",
    "The agent, Tom Hanks, was very rude to me. I want to speak to his manager.",
    "Terrible service. Contact me at 020-1234567. — Frustrated customer",
    "I want to return my order, it does not fit me. 12345 is the order number"]

for i, t in enumerate(tickets_pii):
    print(f"[{i}] {t}")

In [ ]:
# ▶ RUN — regex scrubber: fast and free, catches format-defined PII

PII_PATTERNS = {
    'email':        r'[\w.+-]+@[\w-]+\.[\w.]+',
    'phone_nl':     r'(\+31|0)[\s\-]?[0-9]{1,2}[\s\-]?[0-9]{6,8}',
    'order_number': r'\b(order\s+#?\s*)?\d{5,6}\b',
    'invoice':      r'\binvoice\s+\d{4,6}\b'}

def regex_scrub(text: str) -> str:
    for label, pattern in PII_PATTERNS.items():
        text = re.sub(pattern, f'[{label.upper()}]', text, flags=re.IGNORECASE)
    return text

print(f"{'Original':<80} After regex scrub")
print("-" * 110)
for ticket in tickets_pii:
    scrubbed = regex_scrub(ticket)
    print(f"{ticket[:79]:<80} {scrubbed}")
    print()

print("⚠️  Which names slipped through the regex?")


**What regex catches:** emails, phone numbers, order/invoice numbers with recognisable patterns.

**What regex misses:** names embedded in sentences —
"Maria Jansen", "Jan de Vries", "Emma", "Sophie van der Berg" —
because they look like any other sequence of capitalised words.

This is the fundamental limit of pattern-based scrubbing: it knows *formats*, not *meaning*.



In [ ]:
# ▶ RUN — LLM pass: catch what regex missed
#
# We run regex first (free), then LLM only on the already-scrubbed text.
# This way we minimise how much raw PII the LLM ever sees.

DETECT_PROMPT = """Identify any remaining personal data in this text that should be anonymised.
Types to look for: person name, company name, address, any other identifier.
Return JSON only — no other text:
{{"pii_items": [{{"text": "...", "type": "..."}}]}}
If nothing found return: {{"pii_items": []}}

Text: \"{text}\" """

print("Two-pass scrubbing (regex → LLM):\n")
for i, ticket in enumerate(tickets_pii):
    after_regex = regex_scrub(ticket)
    raw         = call_llm(DETECT_PROMPT.format(text=after_regex), max_tokens=150)
    try:
        detected = json.loads(re.sub(r'```(?:json)?\n?', '', raw).strip())
        items    = detected.get('pii_items', [])
    except json.JSONDecodeError:
        items = []

    if items:
        print(f"[{i}] After regex: {after_regex}")
        print(f"     LLM found:  {items}")
        print()

In [ ]:
# 🎯 YOUR TURN — build the combined scrubber

# Combine the two passes into one function:
#   1. Apply regex_scrub
#   2. Run LLM detection on the result
#   3. Replace each detected PII item with [NAME] or [COMPANY] as appropriate
#   4. Return the fully scrubbed text

# Then verify all 8 tickets are clean.

def full_scrub(text: str) -> str:
    # Step 1
    after_regex = regex_scrub(text)

    # Step 2 - LLM detection on already-scrubbed text
    # YOUR CODE HERE

    # Step 3 - replace each detected item with the appropriate label
    return after_regex  # replace with fully scrubbed version


# Uncomment to test:
# print("Fully scrubbed tickets:")
# for i, ticket in enumerate(tickets_pii):
#     print(ticket)
#     result = full_scrub(ticket)
#     print(f"[{i}] {result}")
#     print()

### 💬 DISCUSS — Exercise 1

- **What still slips through?** "Emma" in ticket 2 might survive because it reads as a common noun
  in some contexts. At what point is residual risk acceptable — and who makes that call?

- **Over-scrubbing tradeoff:** "The agent [NAME] was very rude to me" loses information useful for
  quality monitoring. "My daughter [NAME] liked the product" loses nothing analytical.
  How do you decide what matters to keep?

- **The irony:** we used an LLM to detect PII before sending to an LLM. In production this
  second-pass LLM would be on-premise or a self-hosted model. What would that change in your stack?

- **GDPR in practice:** scrubbing removes PII from the prompt. It does not remove it from
  the LLM provider's logs. What else needs to be in place before you can call this compliant?

> 🎯 **Pattern to remember:**
> - Regex → format-defined PII (email, phone, IBAN, order numbers with known structure)
> - LLM → context-defined PII (names, roles, relationships embedded in sentences)
> - Run regex first: it is free, deterministic, and reduces how much PII the LLM ever sees.

---
## Exercise 2 — Minimum viable context ⏱️ 20 min

You have a quarterly sales dataset. Your LLM pipeline answers business questions over it.
Sending all the raw data works — but at 40 rows it's already hundreds of tokens,
and in production you might have 400,000 rows.

**The question:** how little context can you give the LLM before the answer degrades?

You'll build four progressively smaller representations of the same data,
run the same question through each, and track token count vs answer quality.
The goal is to find the **minimum viable context** — the smallest input that still
produces a useful answer.


In [ ]:
# ▶ RUN — synthetic Q4 2024 sales data

rng = np.random.default_rng(42)

regions    = ["NL", "DE", "BE", "FR"]
categories = ["Electronics", "Clothing", "Food", "Books"]
c_types    = ["B2B", "B2C"]

def create_sales_df(n: int) -> pd.DataFrame:
    return pd.DataFrame({
    "order_id":        range(1001, 1001 + n),
    "date":            pd.date_range("2024-10-01", periods=n, freq="2D").strftime("%Y-%m-%d"),
    "region":          rng.choice(regions, n, p=[0.45, 0.30, 0.15, 0.10]),
    "category":        rng.choice(categories, n, p=[0.35, 0.30, 0.20, 0.15]),
    "customer_type":   rng.choice(c_types, n, p=[0.40, 0.60]),
    "units":           rng.integers(1, 15, n),
    "revenue_eur":     rng.integers(80, 2200, n),
    "discount_pct":    rng.integers(0, 31, n),
    "returned":        rng.choice([True, False], n, p=[0.08, 0.92]),
})

n = 40
sales_df = create_sales_df(n)

print(f"Shape: {sales_df.shape}")
print(f"Total revenue: €{sales_df['revenue_eur'].sum():,}")
print(f"Regions: {sales_df['region'].value_counts().to_dict()}")
print(f"Categories: {sales_df['category'].value_counts().to_dict()}")
print()
print(sales_df.head(6).to_string(index=False))

In [ ]:
# 🎯 YOUR TURN — build four progressively compressed versions of the same data

def create_compressed_versions(df: pd.DataFrame) -> dict[str, str]:

    # Version 1 — full CSV, all rows and columns
    v1_full = df.to_csv(index=False)

    # Version 2 — aggregated by region × category
    agg = (
        df[~df["returned"]]
        .groupby(["...", "..."])    # YOUR CODE HERE
        .agg(                       # Select fitting aggregations for each column (count, sum, mean, etc.)
            orders=("order_id", "..."),
            revenue=("revenue_eur", "..."),
            avg_discount=("discount_pct", "..."),
            b2b_share=("customer_type", lambda x: (x == "B2B").mean().round(2)),
        )
        .reset_index()
    )
    v2_aggregated = agg.to_string(index=False)

    # Version 3 — top-line numbers only
    top_region   = df.groupby("region")["revenue_eur"].sum().idxmax()
    top_category = df.groupby("category")["revenue_eur"].sum().idxmax()
    v3_topline = f"""Q4 2024 Sales Summary — {n} orders
    Total revenue:    €{df['revenue_eur'].sum():,}
    Return rate:      {df['returned'].mean():.0%}
    Avg discount:     {df['discount_pct'].mean():.1f}%
    Top region:       {top_region} (€{df.groupby('region')['revenue_eur'].sum()[top_region]:,})
    Top category:     {top_category} (€{df.groupby('category')['revenue_eur'].sum()[top_category]:,})
    B2B share:        {(df['customer_type']=='B2B').mean():.0%}"""

    # Version 4 — one sentence
    v4_sentence = (
        f"Q4 2024: €{df['revenue_eur'].sum():,} revenue across {n} orders, "
        f"best region {top_region}, best category {top_category}, {df['returned'].mean():.0%} return rate."
    )

    versions = {
        "v1 — full CSV":          v1_full,
        "v2 — region×category":   v2_aggregated,
        "v3 — top-line numbers":  v3_topline,
        "v4 — one sentence":      v4_sentence,
    }

    return versions
    

versions = create_compressed_versions(sales_df)
for label, content in versions.items():
    print(f"{label:30} {len(content):5} chars")

In [ ]:
# 🎯 YOUR TURN — Print the revenue, excluding returns, per region and category to get a feeling for the numbers yourself

# tip: [code].unstack(fill_value=0).style.format("€{:,}")
# YOUR CODE HERE

In [ ]:
# ▶ RUN — same question, four versions, track tokens and answer quality

QUESTION = (
    "Which combination of region and product category should we prioritise in Q1 2025? "
    "Give a specific recommendation with numbers to back it up."
)

results = {}
for label, context in versions.items():
    prompt = f"Data:\n{context}\n\nQuestion: {QUESTION}"
    answer, tokens = call_llm_tracked(prompt)
    results[label] = {"tokens": tokens, "answer": answer}
    print(f"\n{'='*60}")
    print(f"{label}  |  input tokens: {tokens}")
    print(f"{'='*60}")
    print(answer)

In [ ]:
# 🎯 YOUR TURN — find where the answer breaks
#
# Read the four answers above. For each version, assess:
#   - Does it name a specific region AND category?
#   - Does it cite at least one number?
#   - Would you act on this recommendation? Is it too vague, incorrect, or spot on?
#
# Fill in your assessment:

assessment = {
    "v1 — full CSV":         {"region": "?", "category": "?", "numbers": "Y/N", "actionable": "?"},
    "v2 — region×category":  {"region": "?", "category": "?", "numbers": "Y/N", "actionable": "?"},
    "v3 — top-line numbers": {"region": "?", "category": "?", "numbers": "Y/N", "actionable": "?"},
    "v4 — one sentence":     {"region": "?", "category": "?", "numbers": "Y/N", "actionable": "?"}
}

for version, a in assessment.items():
    print(f"{version:30} region: {a['region']:4}, category: {a['category']:14}, numbers: {a['numbers']:4}, actionable: {a['actionable']:4}")

In [ ]:
# ▶ RUN — token cost vs answer completeness at a glance

print(f"{'Version':30} {'Tokens':8} {'Cost index'}")
print("-" * 61)
baseline = results["v1 — full CSV"]["tokens"]
for label, res in results.items():
    t = res["tokens"]
    bar = "█" * int(t / baseline * 20)
    print(f"{label:30} {t:8,}  {bar}")

print()
print("At $0.25 per million input tokens (Haiku pricing):")
for label, res in results.items():
    cost_per_1000 = res["tokens"] / 1_000_000 * 0.25 * 1000
    print(f"  {label:30} ${cost_per_1000:.4f} per 1,000 queries")

### 💬 DISCUSS — Exercise 2

- **Where did quality degrade?** Was it at v3 or v4? Did the model start making up numbers or just become vague?

- **v2 is usually the sweet spot.** Aggregating by the dimensions the question cares about gives the LLM the numbers it needs without exposing individual rows. What determines which dimensions to aggregate on?

- **The v1 problem at scale:** 40 rows is manageable. At 40,000 rows you hit the context window. At 4,000,000 rows you'd need to pre-aggregate anyway. The question is whether you're making that choice *intentionally* or getting forced into it.

- **What v4 loses:** a one-sentence summary tells you the winner but not the margin, the trend, or the second-place candidate. When is that enough?

> 🎯 **Practical rule:** aggregate to the granularity the question requires, not coarser and not finer.
> If the question is about regions, aggregate by region. If it's about region × category, aggregate by both.
> Build the aggregation your business question needs — then add a one-sentence headline at the top.

---
## Exercise 3 — Right context, right question ⏱️ 15 min

You have a wide customer dataset — 15 columns. You are building a pipeline that answers
ad-hoc questions over it. The naive approach: send all 15 columns every time.

The problem: irrelevant columns waste tokens, add noise, and can distract the model.
For a churn question you need recency and NPS. For a revenue question you need basket size
and frequency. Sending everything conflates the two.

**The question:** which columns does *this* question actually need?

**The pattern:** before answering, ask the LLM which columns it needs.
Then fetch only those. Compare answer quality and token cost against sending everything.

The interesting outcome: the LLM sometimes selects columns you wouldn't have chosen,
and sometimes misses columns that domain knowledge says are critical.
That gap is worth understanding.

In [ ]:
# ▶ RUN — wide customer dataset: 15 columns

rng2 = np.random.default_rng(7)
m    = 30

customers_df = pd.DataFrame({
    'customer_id':           range(1, m + 1),
    'days_since_last_order': rng2.integers(1, 180, m),
    'total_orders':          rng2.integers(1, 50, m),
    'avg_order_value_eur':   rng2.integers(20, 300, m),
    'last_nps_score':        rng2.integers(0, 11, m),
    'open_support_tickets':  rng2.integers(0, 5, m),
    'avg_resolution_days':   rng2.integers(1, 14, m),
    'email_open_rate_pct':   rng2.integers(0, 101, m),
    'subscription_months':   rng2.integers(1, 60, m),
    'payment_failures_90d':  rng2.integers(0, 4, m),
    'country':               rng2.choice(['NL', 'DE', 'BE', 'FR'], m),
    'preferred_category':    rng2.choice(['Electronics', 'Clothing', 'Books', 'Food'], m),
    'discount_usage_pct':    rng2.integers(0, 81, m),
    'mobile_app_user':       rng2.choice([True, False], m),
    'account_tier':          rng2.choice(['bronze', 'silver', 'gold'], m, p=[0.5, 0.35, 0.15]),
})

COLUMN_DESCRIPTIONS = {
    'customer_id':           'Unique identifier — not meaningful for analysis',
    'days_since_last_order': 'Days since the customer last placed an order',
    'total_orders':          'Total orders placed since account creation',
    'avg_order_value_eur':   'Average basket size in EUR',
    'last_nps_score':        'Net Promoter Score at last survey (0-10)',
    'open_support_tickets':  'Number of currently open support tickets',
    'avg_resolution_days':   'Average days to resolve a support ticket historically',
    'email_open_rate_pct':   'Percentage of marketing emails opened in last 90 days',
    'subscription_months':   'Months since account creation',
    'payment_failures_90d':  'Failed payment attempts in the last 90 days',
    'country':               'Customer country: NL, DE, BE, FR',
    'preferred_category':    'Product category with highest purchase volume',
    'discount_usage_pct':    'Percentage of orders that used a discount code',
    'mobile_app_user':       'True if at least one order placed via the mobile app',
    'account_tier':          'Loyalty tier: bronze / silver / gold',
}

print(f"Dataset: {customers_df.shape[0]} customers, {customers_df.shape[1]} columns")
print(customers_df.head(3).to_string(index=False))

In [ ]:
# 🎯 YOUR TURN — send only selected columns and compare 
# How would you do compared to the LLM?

QUESTION = (
    "Which customers are most at risk of churning in the next 30 days? "
    "Identify the top 3 warning signs and name 3 specific customer IDs to prioritise."
)


# Version 1
# Choose your own set of columns (what YOU think are the most relevant)
#    and run your version

my_cols             = ['customer_id', '...', '...']
prompt_my           = f"Data:\n{customers_df[my_cols].to_csv(index=False)}\n\nQuestion: {QUESTION}"
answer_my, tok_my   = call_llm_tracked(prompt_my, max_tokens=600)

print(f"Input tokens (my selection): {tok_my:,}")
print()
print(answer_my)

In [ ]:
# ▶ RUN - sending all columns

# Version 2 - all columns

prompt_all          = f"Data:\n{customers_df.to_csv(index=False)}\n\nQuestion: {QUESTION}"
answer_all, tok_all = call_llm_tracked(prompt_all, max_tokens=800)

print(f"Input tokens (all {customers_df.shape[1]} columns): {tok_all:,}")
print()
print(answer_all)

In [ ]:
# ▶ RUN — ask the LLM which columns it actually needs before answering

# Version 3 - LLM selects columns

SELECT_PROMPT = """You are about to answer a business question using a customer dataset.
    Before answering, select only the columns you need. Exclude customer_id from your reasoning
    but always include it for identification.

    Question: {question}

    Available columns:
    {columns}

    Return JSON only:
    {{"needed_columns": ["col1", "col2"], "reason": "one sentence explaining your selection"}}"""

col_list = "\n".join(f"  {col}: {desc}" for col, desc in COLUMN_DESCRIPTIONS.items())
raw      = call_llm(SELECT_PROMPT.format(question=QUESTION, columns=col_list), max_tokens=250)

try:
    sel          = json.loads(re.sub(r'```(?:json)?\n?', '', raw).strip())
    needed_cols  = sel['needed_columns']
    if 'customer_id' not in needed_cols:
        needed_cols = ['customer_id'] + needed_cols
    reason = sel.get('reason', '')
except (json.JSONDecodeError, KeyError):
    print("Could not parse selection. Raw output:")
    print(raw)
    needed_cols, reason = list(customers_df.columns), ''

print(f"LLM selected {len(needed_cols)} of {customers_df.shape[1]} columns:")
for col in needed_cols:
    print(f"  ✓ {col}")
print(f"\nReason: {reason}")

In [ ]:
# Version 3: LLM-selected columns
prompt_sel          = f"Data:\n{customers_df[needed_cols].to_csv(index=False)}\n\nQuestion: {QUESTION}"
answer_sel, tok_sel = call_llm_tracked(prompt_sel, max_tokens=600)

print(f"Input tokens (LLM-selected columns): {tok_sel:,}")
print()
print(answer_sel)

In [ ]:
# Comparison table
print(f"{'Version':<20} {'Cols':>5} {'Tokens':>8}")
print("-" * 38)
print(f"{'All columns':<20} {customers_df.shape[1]:>5} {tok_all:>8,}")
print(f"{'LLM-selected':<20} {len(needed_cols):>5} {tok_sel:>8,}")
print(f"{'My selection':<20} {len(my_cols):>5} {tok_my:>8,}")

### 💬 DISCUSS — Exercise 3
- **Did the LLM pick what you would have picked?** Which columns surprised you —
  either included when you wouldn't have, or missing when you would have?

- **`payment_failures_90d` is a strong churn signal** in subscription businesses.
  Did the LLM select it? If not, what does that tell you about when to override
  the LLM's column selection with domain knowledge?

- **The meta-question cost:** asking which columns to use costs one extra API call
  before every query. When is that worth it? Think about pipelines with many different
  question types hitting the same wide dataset.

- **A caching pattern for production:** after the first run, store the
  question-type → column mapping. Next time the same question type comes in,
  skip the selection call and use the cached set. What could go wrong with this?

> 🎯 **The principle:** every column you include is a claim that it is relevant to this question.
> Irrelevant columns add tokens, can distract the model, and make the prompt harder to audit.
> Decide column inclusion deliberately — the same way you decide which columns go into a dbt model.
> The domain knowledge that doesn't fit in a column description belongs in analysis_guidance.

---
## Exercise 4 — When the LLM transforms the data ⏱️ 15 min

So far you prepared the data before the LLM saw it. This exercise flips the script:
the LLM *is* the transformation step — messy free text in, structured columns out.

Your source system is a marketplace where sellers enter product listings as free text.
Your analytics pipeline needs structured columns: brand, size, category.
An LLM seems like the obvious solution — but its output feeds the next step in the pipeline,
so everything you learned about validation still applies.

**What you'll discover:**
- An LLM handles clean input well — but real data has edge cases
- Asking for JSON doesn't guarantee you get valid JSON
- The production pattern is: extract → validate → retry on failure

Without the validate + retry loop, one bad row silently breaks your pipeline.

In [ ]:
# ▶ RUN — the product listings your pipeline needs to process, 

products = [
    # clean-ish inputs
    "Nike Air Max 90 - white / grey - EU 43 - mens",
    "adidas ultraboost 22 women pink EU 38",
    "PUMA Suede Classic+ Black - Size 42 EU",
    "Reebok black, white or orange, male, size 43/44",
    "New Balance 574 grey - UK 9",
    "Vans Old Skool checkerboard EU 41",
    # edge cases
    "2x Nike Air Force 1 white size 41 + 42",         # two products in one string
    "jordan 1 chicago 10US DS",                       # DS = deadstock, seller jargon
    "running shoe blue XL",                           # no brand, no numeric size
    "adidas yeezy 350 v2 zebra EU43/US9.5/UK8.5",     # three size systems at once
    "shoe",                                           # essentially empty
]

print(f"{len(products)} product listings to process")
for i, p in enumerate(products):
    print(f"  [{i}] {p}")

### Step 1 — Naive extraction: just ask for JSON

In [ ]:
# ▶ RUN — ask the LLM to extract fields, no schema, no instructions

NAIVE_PROMPT = """Extract the following fields from this product listing and return JSON:
brand, color, size_eu, gender, category

Listing: "{listing}"""

print("Testing on the first 4 products:\n")
for product in products[:4]:
    result = call_llm(NAIVE_PROMPT.format(listing=product))
    print(f"Input:  {product}")
    print(f"Output: {result}")
    print()

The clean inputs probably worked, but, even for the clean inputs, notice:

- Is `size_eu` always a number, or sometimes a string like `"43"`?
- Is `color` always a single value, or sometimes a list?
- Does `gender` use consistent values (`mens` vs `men` vs `male`)?

Overall, this inconsistency is the problem. `df["size_eu"].mean()` will fail if half the values are strings.

### Step 2 — Add a schema to the prompt

In [ ]:
# ▶ RUN — constrain the output format with an explicit schema

SCHEMA_PROMPT = """Extract structured data from a product listing.

Return ONLY a JSON object with exactly these fields:
{{
  "brand":    string or null,
  "color":    list of strings (e.g. ["white", "grey"]),
  "size_eu":  number or null  (convert UK/US sizes to EU; if multiple, return the first),
  "gender":   "mens" | "womens" | "unisex" | null,
  "category": "running" | "lifestyle" | "basketball" | "skate" | "other" | null,
  "flags":    list of strings for any issues (e.g. ["multiple_products", "unknown_jargon", "missing_brand"])
}}

Return no other text, no markdown, no explanation.

Listing: "{listing}"""

print("Testing all products with schema prompt:\n")
raw_results = []
for i, product in enumerate(products):
    result = call_llm(SCHEMA_PROMPT.format(listing=product))
    raw_results.append(result)
    print(f"[{i}] {product[:50]:50} →  {result}")

### Step 3 — Validate: don't trust the output

In [ ]:
# ▶ RUN — parse and validate every result

REQUIRED_FIELDS  = {"brand", "color", "size_eu", "category", "flags"}
VALID_CATEGORIES = {"running", "lifestyle", "basketball", "skate", "other", None}
VALID_GENDERS    = {"mens", "womens", "unisex", None}

def parse_llm_json(text: str) -> dict | None:
    text = re.sub(r"```(?:json)?\n?", "", text).strip().rstrip("`")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def validate(result: dict | None) -> tuple[bool, str]:
    if result is None:
        return False, "not valid JSON"
    missing = REQUIRED_FIELDS - set(result.keys())
    if missing:
        return False, f"missing fields: {missing}"
    size = result.get("size_eu")
    if size is not None:
        try:
            if not (30 <= float(size) <= 55):
                return False, f"size_eu out of range: {size}"
        except (TypeError, ValueError):
            return False, f"size_eu not numeric: {size!r}"
    if result.get("category") not in VALID_CATEGORIES:
        return False, f"invalid category: {result['category']!r}"
    if not isinstance(result.get("color"), list):
        return False, f"color must be a list, got: {type(result.get('color')).__name__}"
    return True, "ok"


parsed   = [parse_llm_json(r) for r in raw_results]
statuses = [validate(p) for p in parsed]

print(f"{'#':3} {'Product':45} {'Valid':6} {'Reason'}")
print("-" * 85)
for i, (product, (ok, reason)) in enumerate(zip(products, statuses)):
    mark = "✓" if ok else "✗"
    print(f"[{i}] {product[:44]:44} {mark:6} {reason}")

passed = sum(ok for ok, _ in statuses)
print(f"\n{passed}/{len(products)} passed validation")

### Step 4 — Retry: send the error back 

When validation fails, most pipelines either crash or silently skip the row.
A better pattern: tell the LLM what it got wrong and ask it to fix it.

This works better than retrying with the same prompt — the error message gives the model
the information it needs to correct its specific mistake.

In [ ]:
# 🎯 YOUR TURN — complete the retry function 

# The function should:
#   1. Try extraction with SCHEMA_PROMPT
#   2. Parse and validate the result
#   3. If invalid, build a correction prompt that includes:
#      - the original listing
#      - the bad output the LLM returned
#      - the specific validation error
#   4. Try once more with the correction prompt
#   5. Return the result and whether it needed a retry

CORRECTION_PROMPT = """You returned invalid JSON for a product listing. Fix it.

    Original listing: "{listing}"
    Your previous output: {bad_output}
    Validation error: {error}

    Return ONLY the corrected JSON object. Same schema as before."""

def extract_with_retry(listing: str) -> tuple[dict | None, bool, str]:
    """
    Returns: (parsed_result, needed_retry, final_status)
    """
    # Attempt 1
    raw = call_llm(SCHEMA_PROMPT.format(listing=listing))
    parsed = parse_llm_json(raw)
    ok, reason = validate(parsed)
    if ok:
        return parsed, False, "ok"

    # Attempt 2 — send the error back
    # YOUR CODE HERE
    # correction = call_llm(CORRECTION_PROMPT.format(...))
    # parsed2 = parse_llm_json(correction)
    # ok2, reason2 = validate(parsed2)
    # return parsed2, True, reason2

# Uncomment to run once implemented:
# print(f"{'#':3} {'Product':45} {'Retry':6} {'Status'}")
# print("-" * 80)
# for i, product in enumerate(products):
#     result, retried, status = extract_with_retry(product)
#     mark = "↩" if retried else " "
#     ok = "✓" if status == "ok" else "✗"
#     print(f"[{i}] {product[:44]:44} {mark:6} {ok} {status}")
#     print()

### 💬 DISCUSS — Exercise 4

- **The edge cases:** listings 6–10 represent real patterns in marketplace data. Which ones did the retry loop fix? Which ones are fundamentally unfixable by an LLM?

- **The `flags` field:** we asked the LLM to self-report issues (`multiple_products`, `unknown_jargon`). How reliable is self-reporting? When would you trust it, and when would you add your own rule-based checks?

- **Cost:** each retry doubles the API calls for failed rows. If 20% of your listings fail the first attempt, what's the cost impact at 100,000 listings/day?

- **Alternative:** for the `size_eu` conversion specifically, a lookup table (US→EU, UK→EU) would be faster, cheaper, and more reliable than an LLM. When is an LLM the right tool here, and when isn't it?

> 🎯 **Rule of thumb:** if the transformation can be expressed as a lookup table or regex, use that.
> Use an LLM when the input is too varied and ambiguous for rules — and always validate the output.

---
## Exercise 5 — Numbers they can trust ⏱️ 15 min

A regional sales manager uses a Slack bot: *"What was Electronics revenue in the Netherlands in October,
and how does that compare to Clothing?"*

Your pipeline dumps 50 order rows into the prompt and asks the LLM to answer.
It often returns plausible-sounding numbers that are **wrong** — arithmetic errors,
wrong filters, or rows double-counted.

**The problem:** LLMs are good at language, unreliable at arithmetic over tables.
Business users will act on the answer if it looks confident.

**The fix — text-to-SQL in three steps:**
1. LLM reads the question + schema → writes a SQL query *(language task: LLM is good at this)*
2. Database executes the query → returns exact numbers *(arithmetic: always deterministic)*
3. LLM reads the result → explains it in plain language *(language task: LLM is good at this)*

This is the pattern behind production text-to-SQL systems and LLM tool calling.
The database does the math. The LLM handles phrasing and follow-up questions.
You never need to anticipate which specific numbers the user will ask for.

In [ ]:
# ▶ RUN — load October orders into SQLite

rng = np.random.default_rng(77)
regions    = ["DE", "NL", "BE"]
categories = ["Electronics", "Clothing", "Food"]

n_orders = 50
orders_oct_df = pd.DataFrame({
    "order_id":     range(2001, 2001 + n_orders),
    "order_date":   pd.date_range("2024-10-01", periods=n_orders, freq="12h").strftime("%Y-%m-%d"),
    "region":       rng.choice(regions, n_orders, p=[0.40, 0.35, 0.25]),
    "category":     rng.choice(categories, n_orders, p=[0.45, 0.35, 0.20]),
    "revenue_eur":  rng.integers(90, 1800, n_orders),
    "returned":     rng.choice([0, 1], n_orders, p=[0.88, 0.12]),
})

conn = sqlite3.connect(":memory:")
orders_oct_df.to_sql("orders", conn, index=False, if_exists="replace")

print(orders_oct_df.head(5).to_string(index=False))
print(f"\nDate range: {orders_oct_df['order_date'].min()} to {orders_oct_df['order_date'].max()}")
print(f"Total rows: {len(orders_oct_df)} | Returned: {orders_oct_df['returned'].sum()}")

In [ ]:
# ▶ RUN — define the question and schema description

QUESTION = (
    "What was Electronics revenue, excluding returned orders, in the Netherlands in October, "
    "and how does that compare to Clothing? "
    "Give exact EUR amounts and say which category performed better."
)

SCHEMA = """Table: orders
  order_id     INTEGER
  order_date   TEXT     -- YYYY-MM-DD, all rows are October 2024
  region       TEXT     -- DE, NL, BE
  category     TEXT     -- Electronics, Clothing, Food
  revenue_eur  INTEGER
  returned     INTEGER  -- 1 = returned, 0 = not returned"""

print("Schema loaded. Question:")
print(QUESTION)

In [ ]:
# ▶ RUN — naive: LLM calculates from raw rows

naive_prompt = f"""Data (October orders):
    {orders_oct_df.to_csv(index=False)}

    Question: {QUESTION}

    Exclude returned orders from the revenue. Answer in 3-4 sentences."""

answer_naive, tok_naive = call_llm_tracked(naive_prompt, max_tokens=350)

# Ground truth via SQL so we can check the LLM's answer
ground_truth = pd.read_sql("""
    SELECT category, SUM(revenue_eur) AS revenue_eur
    FROM orders
    WHERE region = 'NL' AND returned = 0
    GROUP BY category
    ORDER BY revenue_eur DESC
""", conn)

print(f"NAIVE (LLM does the math) | input tokens: {tok_naive}")
print("=" * 60)
print(answer_naive)
print()
print("Ground truth (SQL):")
print(ground_truth.to_string(index=False))
print("\n☝ Did the LLM get the right numbers?")

In [ ]:
# ▶ RUN — text-to-SQL: Step 1 (LLM writes SQL) + Step 2 (DB executes)

# Step 1 — LLM writes SQL (language task)
sql_prompt = f"""Write a single SQLite SELECT query to answer the question below.

Schema:
{SCHEMA}

Return ONLY the SQL query. No explanation, no markdown, no code fences.

Question: {QUESTION}"""

generated_sql = call_llm(sql_prompt, max_tokens=350)
print("Step 1 — LLM-generated SQL:")
print(generated_sql)
print()

# Step 2 — database executes it (deterministic arithmetic, no LLM involved)
try:
    result_df = pd.read_sql(generated_sql, conn)
    print("Step 2 — Query result:")
    print(result_df.to_string(index=False))
    query_ok = True
except Exception as e:
    print(f"SQL error — fix the query and re-run: {e}")
    result_df = pd.DataFrame()
    query_ok = False

In [ ]:
# ▶ RUN — Step 3: LLM narrates the result (language task)

if query_ok:
    narrate_prompt = f"""A business user asked: "{QUESTION}"

        The database returned this result:
        {result_df.to_string(index=False)}

        Explain this to a regional sales manager in 2-3 sentences.
        Use only the numbers in the table above — do not recalculate."""

    answer_grounded, tok_grounded = call_llm_tracked(narrate_prompt, max_tokens=200)

    print(f"TEXT-TO-SQL (DB computes, LLM explains) | input tokens: {tok_grounded}")
    print("=" * 60)
    print(answer_grounded)
    print()
    
# LLM wrote the query → DB computed the numbers → LLM explained them
# The model never did arithmetic. Every number comes from the database.

In [ ]:
# 🎯 YOUR TURN
#
# Change MY_QUESTION to anything about this table and run the cell again.
# The three-step pipeline handles any question without any code changes.

MY_QUESTION = "" # YOUR CODE HERE, e.g.: "Which region had the highest total revenue in October? Exclude returns."

sql_prompt = f"""Write a single SQLite SELECT query to answer the question below.
    Return ONLY the SQL query. No explanation, no markdown, no code fences.
    Schema: {SCHEMA}
    Question: {MY_QUESTION}"""


my_sql = call_llm(sql_prompt, max_tokens=400)

print(f"\n🎯 YOUR TURN — generated SQL for: {MY_QUESTION}")
print(my_sql)

try:
    my_result = pd.read_sql(my_sql, conn)
    print(my_result.to_string(index=False))
    print()
    print(call_llm(
        f"Question: {MY_QUESTION}\n\nResult:\n{my_result.to_string(index=False)}\n\n"
        f"Answer in 1-2 sentences. Use only the numbers above.",
        max_tokens=250,
    ))
except Exception as e:
    print(f"SQL error: {e}")

### 💬 DISCUSS — Exercise 5

- **Did the naive answer match the ground truth?** If not, would a busy manager have noticed?

- **The three-step split:**
  - LLM writes SQL → language task, LLM is reliable
  - DB executes SQL → arithmetic, always correct
  - LLM narrates result → language task, LLM is reliable

  What breaks if you collapse this to one step and let the LLM answer from raw rows?

- **Token count:** the grounded prompt sends only the query result instead of the full CSV. Cheaper or more expensive than the naive approach?

- **What if the LLM generates bad SQL?** How do you handle that in production?
  (retry with the error in the prompt, ask for clarification, human-in-the-loop)

- **Generalising:** this notebook uses SQLite. In production the same pattern works with Snowflake,
  BigQuery, PostgreSQL — anything that can run a query. The data engineer's job is to keep schema
  documentation accurate so the LLM writes correct SQL.

> 🎯 **Practical rule:** let SQL/Python compute; let the LLM communicate.
> A number that drives a business decision should come from deterministic code — not from the model's mental math.

---
# Extra exercises for a later moment

## Exercise 6 — Signal that survives ⏱️ 15 min

You have 80 support tickets with a skewed theme distribution — shipping delays are 40%,
account issues only 8%. To decrease token usage, your pipeline batches tickets, summarises each batch, then combines
the summaries. Sounds efficient.

**The problem:** naive map-reduce treats every batch summary as equally weighted.
A rare theme in one batch can look as important as a dominant theme in another.
Summarise too aggressively and you lose class imbalance — the signal that should drive
prioritisation.

**What you'll build:** count frequencies first (no LLM), sample proportionally,
then pass explicit counts to the LLM so the ranking survives the reduction.

In [ ]:
# ▶ RUN — generate 80 synthetic support tickets with a skewed distribution

rng = np.random.default_rng(99)

THEME_WEIGHTS = {
    'shipping_delay':  0.40,
    'wrong_item':      0.25,
    'product_quality': 0.15,
    'payment_issue':   0.12,
    'account_access':  0.08,
}

TEMPLATES = {
    'shipping_delay': [
        'My order #{oid} placed {days} days ago still has not arrived.',
        'Package stuck in transit for {days} days. Tracking shows no movement.',
        'Delivery is {days} days late with no notification from the carrier.',
        'Where is my parcel? Expected last week, nothing yet. Order #{oid}.',
        'Second delay on order #{oid}. I need this urgently.',
    ],
    'wrong_item': [
        'Received wrong item in my package for order #{oid}. I did not order this.',
        'I ordered a blue one but received a red one. Please correct this.',
        'The product in my box does not match my order confirmation #{oid}.',
        'Wrong size delivered. I ordered L, received S.',
        'Completely different product sent. Nothing like what I ordered.',
    ],
    'product_quality': [
        'Product broke after {days} days of normal use. Very poor quality.',
        'Item arrived already damaged. Packaging was crushed.',
        'Quality much worse than shown in the website photos.',
        'Colour faded completely after the first wash. Not as described.',
        'Stopped working after one week. Clearly a manufacturing defect.',
    ],
    'payment_issue': [
        'Charged twice for order #{oid}. Please refund the duplicate charge.',
        'My refund for the return from {days} days ago has not arrived yet.',
        'Discount code did not apply at checkout but I was charged full price.',
        'Wrong amount on my invoice. I was overcharged.',
        'Payment failed but money still left my bank account.',
    ],
    'account_access': [
        'Cannot log in. Password reset emails are not arriving.',
        'My account appears locked after three failed login attempts.',
        'Two-factor authentication is broken — SMS code never arrives.',
        'Cannot access my order history. Page shows a server error.',
        'I accidentally created two accounts and need them merged into one.',
    ],
}

themes  = list(THEME_WEIGHTS.keys())
weights = list(THEME_WEIGHTS.values())
n       = 80

chosen  = rng.choice(themes, n, p=weights)
tickets = []
for theme in chosen:
    tmpl = rng.choice(TEMPLATES[theme])
    text = tmpl.format(oid=int(rng.integers(10000, 99999)), days=int(rng.integers(3, 18)))
    tickets.append({'theme': theme, 'text': text})

tickets_df = pd.DataFrame(tickets)

print("Ground truth distribution (what the LLM should ideally report):")
print(tickets_df['theme'].value_counts().to_string())
print(f"\nTotal tickets: {len(tickets_df)}")

#### Step a) Naive map-reduce: summarise batches, then summarise summaries

In [ ]:
# ▶ RUN — batch into groups of 10, summarise each, then combine

BATCH_PROMPT = """Analyse these customer support tickets and list the main themes you see.
Be concise — one line per theme, no more than 5 themes.

Tickets:
{tickets}"""

COMBINE_PROMPT = """You have summaries from {n} batches of support tickets.
Identify the top 5 recurring themes across all batches ranked by how commonly they appear, and state how commonly they appear.

Batch summaries:
{summaries}"""

BATCH_SIZE = 10
batches    = [tickets_df['text'].tolist()[i:i+BATCH_SIZE]
              for i in range(0, len(tickets_df), BATCH_SIZE)]

print(f"Summarising {len(batches)} batches of {BATCH_SIZE} tickets each.")
batch_summaries = []
for i, batch in enumerate(batches):
    ticket_text = "\n".join(f"- {t}" for t in batch)
    summary     = call_llm(BATCH_PROMPT.format(tickets=ticket_text), max_tokens=120)
    batch_summaries.append(summary)
    print(f"Batch {i+1}: {summary[:120].replace(chr(10), ' ')}...")

combined = "\n\n".join(f"[Batch {i+1}]\n{s}" for i, s in enumerate(batch_summaries))

print("\n" + "=" * 60)
print("NAIVE MAP-REDUCE RESULT")
naive_result = call_llm(COMBINE_PROMPT.format(n=len(batches), summaries=combined), max_tokens=300)
print(naive_result)

#### What just happened?

The naive result likely shows themes as roughly equally important, which are not.

Look at the ground truth: shipping delays appear in **40%** of tickets.
Account access issues appear in only **8%**.

The naive approach lost this — each batch summary carries equal weight in the final combination,
so a batch with 8 shipping tickets contributes the same signal as a batch with 1 shipping ticket.

**Before building a smarter approach — count first.**

In [ ]:
# ▶ RUN — count keyword frequencies across all tickets (Python only, no LLM)

KEYWORDS = {
    'shipping_delay':  ['arrived', 'late', 'delay', 'transit', 'tracking', 'parcel', 'delivery'],
    'wrong_item':      ['wrong', 'incorrect', 'different product', 'did not order'],
    'product_quality': ['broke', 'broken', 'damaged', 'quality', 'defect', 'faded', 'poor'],
    'payment_issue':   ['charged', 'refund', 'payment', 'invoice', 'discount', 'overcharged'],
    'account_access':  ['login', 'password', 'account', 'locked', 'authentication'],
}

def classify_ticket(text: str) -> str:
    t      = text.lower()
    scores = {theme: sum(kw in t for kw in kws) for theme, kws in KEYWORDS.items()}
    best   = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'other'

tickets_df['detected_theme'] = tickets_df['text'].apply(classify_ticket)
freq = tickets_df['detected_theme'].value_counts()

print("Keyword-based frequency estimate:")
print((freq / len(tickets_df) * 100).round(1).astype(str).add('%').to_string())
print()
print("vs ground truth:")
gt  = tickets_df['theme'].value_counts()
cmp = pd.DataFrame({'detected %': (freq / len(tickets_df) * 100).round(1),
                    'actual %':   (gt  / len(tickets_df) * 100).round(1)}).fillna(0)
print(cmp.to_string())

In [ ]:
# 🎯 YOUR TURN — sample proportionally from each theme  ⏱️ 8 min
#
# The frequency count shows the real distribution.
# Build a representative sample: pick tickets in proportion to their detected frequency,
# so the LLM receives context that reflects the actual distribution.
#
# Target: SAMPLE_TOTAL = 12 representative tickets total
# e.g. if shipping is 40%: include ~5 shipping tickets out of 12
#
# Hint: for each theme, compute  n_to_sample = round(freq[theme] / total * SAMPLE_TOTAL)
#       then sample that many rows from tickets_df[tickets_df['detected_theme'] == theme]
#       Take into account each theme must have at least 1 ticket, and you cannot sample more than available.

SAMPLE_TOTAL = 12

samples = []
total = len(tickets_df)

# for theme, count in freq.items():
    # YOUR CODE HERE — build a DataFrame called `representative`

    # samples.append( 
    #     .sample(n=n_to_sample, random_state=42)
    # )

# representative = pd.concat(samples).reset_index(drop=True)

# When done, uncomment to verify:
# print(f"Representative sample: {len(representative)} tickets")
# print(representative['detected_theme'].value_counts().to_string())

In [ ]:
# ▶ RUN — frequency-aware summarisation
# Run after the YOUR TURN cell above defines `representative`.

FREQ_AWARE_PROMPT = """You are analysing customer support tickets to identify recurring themes.

The sample below was selected proportionally from the full dataset of {total} tickets.
Each group header shows how many tickets out of {total} belong to that theme.
Use these counts to rank themes by actual frequency.

{context}

Identify the top 5 themes ranked by frequency. For each, state the approximate share of tickets."""

# Uncomment once `representative` is defined:
# context_lines = []
# for theme, grp in representative.groupby('detected_theme'):
#     count = int(freq.get(theme, 0))
#     pct   = round(count / len(tickets_df) * 100)
#     context_lines.append(f"\n[{theme} | {count} of {len(tickets_df)} tickets ({pct}%)]")
#     for _, row in grp.iterrows():
#         context_lines.append(f"  - {row['text']}")
#
# freq_aware_result = call_llm(
#     FREQ_AWARE_PROMPT.format(
#         total=len(tickets_df),
#         context="\n".join(context_lines),
#     ),
#     max_tokens=400,
# )
# print("=" * 60)
# print("FREQUENCY-AWARE RESULT")
# print("=" * 60)
# print(freq_aware_result)
# print()
# print("Ground truth for comparison:")
# print((tickets_df['theme'].value_counts() / len(tickets_df) * 100).round(1).astype(str).add('%'))

### 💬 DISCUSS — Exercise 6

- **Did the naive result get the ranking right?** Which themes were over- or under-represented
  compared to the ground truth?

- **The keyword classifier is imperfect** — some tickets will be misclassified.
  How wrong does the frequency estimate need to be before it misleads the LLM?
  Does it matter if shipping is 40% vs 38%?

- **This pattern generalises:** map-reduce loses frequency whenever intermediate summaries
  are treated as equally weighted. Where else might this happen in your pipelines?
  (Aggregating feedback by region, summarising weekly reports into a monthly view.)

- **Alternative approaches and their tradeoffs:**

  | Approach | Accuracy | Cost | Speed |
  |----------|----------|------|-------|
  | Keyword count (this exercise) | Good enough for ranking | Free | Fast |
  | LLM classify each ticket | High | Expensive (N API calls) | Slow |
  | Embeddings + clustering | High | Moderate | Moderate |

> 🎯 **Rule of thumb:** whenever you reduce N items into a summary that drives a decision,
> ask whether frequency information survives the reduction. If it doesn't, add explicit counts
> before the final LLM call.

## Exercise 7 — Speak their language ⏱️ 15 min
### Adding a semantic layer and business knowledge

Your company rolled out a self-service analytics chatbot.
A sales director types: *“How many customers churned last month, and what’s our active subscriber base?”*

The warehouse table has column names only engineers recognise:
`acct_stat_cd`, `chrn_flg`, `dunning_lvl`.

**The problem:** without a business glossary, the LLM has to guess which column matches the question.
It will likely count `acct_stat_cd = 'C'` (cancelled) instead of `chrn_flg = 1`
(Finance’s definition: completed cancellation, notice period finished).

**What you’ll build:** a glossary that maps business terms → columns + rules,
and see how much it changes the answer.

In [ ]:
# ▶ RUN — subscriber snapshot (cryptic column names, intentional)

rng6 = np.random.default_rng(6)
n_subs = 15

statuses = rng6.choice(["A", "C", "P"], n_subs, p=[0.62, 0.28, 0.10])
chrn_flg = [
    1 if (s == "C" and rng6.random() < 0.55) else 0
    for s in statuses
]

subscribers_df = pd.DataFrame({
    "cust_id":       range(5001, 5001 + n_subs),
    "acct_stat_cd":  statuses,
    "last_bill_dt":  pd.date_range("2024-11-01", periods=n_subs, freq="2D").strftime("%Y-%m-%d"),
    "mrr_eur":       rng6.integers(15, 250, n_subs),
    "chrn_flg":      chrn_flg,
    "dunning_lvl":   rng6.integers(0, 4, n_subs),
})

# Note: chrn_flg description is intentionally vague — realistic warehouse documentation
COLUMN_DESCRIPTIONS_TECH = {
    "cust_id":      "Customer primary key",
    "acct_stat_cd": "Account status code: A=active, C=cancelled, P=pending cancellation",
    "last_bill_dt": "Date of most recent successful invoice",
    "mrr_eur":      "Monthly recurring revenue in EUR",
    "chrn_flg":     "Binary flag set by Finance ETL",  # ← vague on purpose: what does it mean?
    "dunning_lvl":  "Collections stage 0-3 (0 = none)",
}

finance_churned = int(subscribers_df["chrn_flg"].sum())
finance_active  = int((subscribers_df["acct_stat_cd"] == "A").sum())
naive_cancelled = int((subscribers_df["acct_stat_cd"] == "C").sum())

print(subscribers_df.head(6).to_string(index=False))
print()
print("Ground truth — Finance definitions:")
print(f"  churned  (chrn_flg=1):         {finance_churned}")
print(f"  active   (acct_stat_cd='A'):   {finance_active}")
print(f"  cancelled (acct_stat_cd='C'):  {naive_cancelled}  ← often confused with churned")


In [ ]:
# ▶ RUN — compare: technical descriptions vs business glossary

QUESTION_GLOSSARY = (
    "How many customers churned last month, and how many active subscribers do we have? "
    "Use the data to answer with specific counts."
)

# — Prompt 1: only technical column descriptions —
col_desc = "\n".join(f"  {col}: {desc}" for col, desc in COLUMN_DESCRIPTIONS_TECH.items())
naive_prompt = f"""You are a business analytics assistant.

    Dataset columns:
    {col_desc}

    Data:
    {subscribers_df.to_csv(index=False)}

    Question: {QUESTION_GLOSSARY}

    Answer in 2-3 sentences with the two counts."""

# — Prompt 2: business glossary with explicit rules —
BUSINESS_GLOSSARY = {
    "churned (last month)": {
        "column": "chrn_flg",
        "definition": (
            "Count rows where chrn_flg = 1. Finance definition: customer completed "
            "cancellation during the reporting month. Do NOT use acct_stat_cd='C' — "
            "that includes cancellations still in notice period."
        ),
    },
    "active subscriber": {
        "column": "acct_stat_cd",
        "definition": "Count rows where acct_stat_cd = 'A'. Currently billed and entitled to service.",
    },
}

glossary_text = "\n".join(
    f"- **{term}** → use `{meta['column']}`: {meta['definition']}"
    for term, meta in BUSINESS_GLOSSARY.items()
)

glossary_prompt = f"""You are a business analytics assistant.

BUSINESS GLOSSARY (authoritative — follow these definitions exactly):
    {glossary_text}

    Dataset:
    {subscribers_df.to_csv(index=False)}

    Question: {QUESTION_GLOSSARY}

    Answer in 2-3 sentences. State which column you counted for each metric."""

# — Run both and compare —
answer_naive    = call_llm(naive_prompt, max_tokens=400)
answer_glossary = call_llm(glossary_prompt, max_tokens=400)

print("WITHOUT GLOSSARY (technical descriptions only)")
print("-" * 55)
print(answer_naive)
print()
print("WITH BUSINESS GLOSSARY")
print("-" * 55)
print(answer_glossary)
print()
print(f"✔ Finance expects: churned={finance_churned}, active={finance_active}")
print(f"  (naive reader might count cancelled={naive_cancelled} as churned)")


In [ ]:
# 🎯 YOUR TURN — extend the glossary  ⏱️ 8 min
#
# A stakeholder now asks: "How many subscribers are at risk in collections?"
#
# 1. Add a glossary entry for 'at risk' that maps it to the right column
#    and warns against confusing it with churn.
# 2. Build a prompt using your extended glossary.
# 3. Compare the LLM's count to the Python ground truth below.

QUESTION_AT_RISK = "How many subscribers are at risk in collections right now?"

ground_truth_at_risk = int((subscribers_df["dunning_lvl"] >= 2).sum())
print(f"Ground truth (dunning_lvl >= 2): {ground_truth_at_risk}")

# YOUR CODE HERE
# MY_GLOSSARY = {**BUSINESS_GLOSSARY, "at risk (collections)": { ... }}
# prompt = ...
# answer = call_llm(prompt, max_tokens=200)
# print(answer)


### 💬 DISCUSS — Exercise 7

- **Did the naive answer confuse churn with cancelled?** How far off was it from Finance's `chrn_flg` count?

- **Who owns the glossary?** Should it live in dbt docs, a Confluence page, or code next to the chatbot prompt? What happens when Finance changes the churn definition?

- **Glossary vs column descriptions:** Exercise 4 used per-column descriptions. This exercise adds *business terms* that may span multiple columns or require a rule. When do you need which?

- **Stale glossary is worse than no glossary** — the model sounds confident while applying last quarter's definition. How would you version and audit glossary changes?

> 🎯 **Practical rule:** whenever a business user can ask a question in words that don't appear in your schema,
> attach an explicit term → column → definition mapping. Treat it like a semantic layer for the LLM.